# Dealer Matching on Snowflake — Workshop Notebook
Pipeline steps 1-4 (Collect -> Locate -> Enrich -> Resolve) on a sample of real crawled dealers.

**Before running:**
1. Upload the two files to the stage (Block 0 shows the Snow CLI commands).
2. Have your Google Maps and Gemini API keys ready (pasted in the Block 2 and Block 3 integration cells).
3. Keep `SAMPLE_SIZE` small for the first run: every API call is billed by Google.

## Parameters
`SAMPLE_SIZE` limits the number of API calls (Google Maps + Gemini cost money). Start small.

In [ ]:
SAMPLE_SIZE = 10  # number of dealers sent to the APIs

## Block 0 — Setup

In [ ]:
%%sql -r setup_ctx
-- Everything lives in a dedicated database: drop it after the workshop to clean up.
-- SYSADMIN owns the objects; ACCOUNTADMIN is only needed for the integrations (blocks 2 and 3).
USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS WORKSHOP_DB;
USE DATABASE WORKSHOP_DB;
USE SCHEMA PUBLIC;

-- XSMALL is enough: the heavy work is done by the external APIs, not by the warehouse.
-- AUTO_SUSPEND = 60 s so the warehouse stops costing credits when idle.
CREATE WAREHOUSE IF NOT EXISTS WORKSHOP_WH
    WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE WORKSHOP_WH;

-- Internal stage = Snowflake-managed file storage. It replaces the local folders of the Linux server.
-- Upload the files from your terminal with Snow CLI (run from the repo root):
--   snow stage copy data/dealers_sample.csv @WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE
--   snow stage copy prompts/gemini_search_agency.txt @WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE/prompts
CREATE STAGE IF NOT EXISTS WORKSHOP_STAGE;
LIST @WORKSHOP_STAGE;   -- you should see both files

In [ ]:
%%sql -r create_dealers
-- DEALERS = step 1 output (Collect): one row per dealer per marketplace site.
-- Same name and key as the target architecture: (AGENCY_ID, SITE).
-- Note: Snowflake does not enforce PRIMARY KEY uniqueness (only NOT NULL); it documents the model.
CREATE OR REPLACE TABLE DEALERS (
    AGENCY_ID       VARCHAR,
    SITE            VARCHAR,
    AGENCY_NAME     VARCHAR,
    ADDRESS         VARCHAR,
    CITY            VARCHAR,
    ZIP_CODE        VARCHAR,
    SIRET           VARCHAR,
    MINISITE_URL    VARCHAR,
    GOOGLE_PLACE_ID VARCHAR,   -- filled in step 2
    JORECA_ID       VARCHAR,   -- filled in step 6 (out of scope)
    PRIMARY KEY (AGENCY_ID, SITE)
);

In [ ]:
%%sql -r load_dealers
-- Load the crawl sample from the stage. Equivalent of the INSERTs into MariaDB.
-- The CSV columns are SITE, AGENCY_ID, ... so we list the target columns in that order.
-- EMPTY_FIELD_AS_NULL turns empty strings (e.g. missing SIRET) into real NULLs.
COPY INTO DEALERS (SITE, AGENCY_ID, AGENCY_NAME, ADDRESS, CITY, ZIP_CODE, SIRET, MINISITE_URL)
FROM @WORKSHOP_STAGE/dealers_sample.csv
FILE_FORMAT = (TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1 EMPTY_FIELD_AS_NULL=TRUE);

In [ ]:
%%sql -r create_targets
-- Empty tables for steps 2 and 3, same names as the target architecture.
-- DEALER_GOOGLE_MAP: rolling cache keyed by GOOGLE_PLACE_ID -> a place is fetched once and reused every month.
-- DEALER_ADDRESS_STANDARD: per-period workspace (one row per AGENCY_ID, SITE) holding the Gemini output.
CREATE OR REPLACE TABLE DEALER_GOOGLE_MAP (
    GOOGLE_PLACE_ID VARCHAR PRIMARY KEY,
    AI_AGENCY_NAME  VARCHAR,
    GOOGLE_JSON     VARIANT,
    CREATED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE OR REPLACE TABLE DEALER_ADDRESS_STANDARD (
    AGENCY_ID           VARCHAR,
    SITE                VARCHAR,
    GOOGLE_PLACE_ID     VARCHAR,
    JORECA_ID           VARCHAR,
    IS_REUSED           BOOLEAN DEFAULT FALSE,
    AI_JSON             VARIANT,
    AI_STANDARD_ADDRESS VARCHAR,
    CREATED_AT          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY (AGENCY_ID, SITE)
);

In [ ]:
%%sql -r check_load
-- Expected: DEALERS = 200, the two other tables = 0 at this point.
SELECT 'DEALERS' AS TBL, COUNT(*) AS N FROM DEALERS
UNION ALL SELECT 'DEALER_GOOGLE_MAP', COUNT(*) FROM DEALER_GOOGLE_MAP
UNION ALL SELECT 'DEALER_ADDRESS_STANDARD', COUNT(*) FROM DEALER_ADDRESS_STANDARD;

## Block 1 — Collect: explore DEALERS

In [ ]:
%%sql -r dealers_preview
-- Raw crawl data: look at name and address quality.
SELECT * FROM DEALERS LIMIT 10;

In [ ]:
%%sql -r dealers_by_site
-- Which marketplaces the sample comes from.
SELECT SITE, COUNT(*) AS N FROM DEALERS GROUP BY SITE ORDER BY N DESC;

In [ ]:
%%sql -r siret_coverage
-- How many dealers already have a SIRET from the crawl (the rest must be found by Gemini).
SELECT COUNT(*) AS TOTAL,
       COUNT(SIRET) AS WITH_SIRET,
       ROUND(COUNT(SIRET) / COUNT(*) * 100, 1) AS SIRET_PCT
FROM DEALERS;

In [ ]:
%%sql -r address_issues
-- Typical issues: country prefix in the address (FR-74100), underscores in city names, missing zip.
SELECT AGENCY_NAME, ADDRESS, CITY, ZIP_CODE
FROM DEALERS
WHERE ADDRESS ILIKE 'FR-%' OR CITY LIKE '%\\_%' OR ZIP_CODE IS NULL
LIMIT 20;

## Block 2 — Locate: Google Maps via External Access Integration
Paste the Google Maps key in the cell below (ACCOUNTADMIN is needed for network rule / integration).
Note: the UDF uses the Places API endpoints `findplacefromtext` + `details` — they must be enabled on the Google Cloud project.

In [ ]:
%%sql -r eai_google
-- Snowflake blocks all outbound network calls by default. Three objects open a controlled path:
--   1. NETWORK RULE: which hosts are reachable (only Google Maps here)
--   2. SECRET: the API key, stored encrypted; code reads it at runtime, it is never visible in SQL
--   3. EXTERNAL ACCESS INTEGRATION: binds rule + secret; UDFs must reference it explicitly
-- Requires ACCOUNTADMIN (or CREATE INTEGRATION privilege). Run once.
-- On the Google Cloud project, the Places API (findplacefromtext + details endpoints) must be enabled.
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_RULE
    MODE = EGRESS TYPE = HOST_PORT
    VALUE_LIST = ('maps.googleapis.com');
CREATE OR REPLACE SECRET WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY
    TYPE = GENERIC_STRING
    SECRET_STRING = 'PASTE_GOOGLE_MAPS_KEY_HERE';
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION GOOGLE_MAPS_EAI
    ALLOWED_NETWORK_RULES = (WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_RULE)
    ALLOWED_AUTHENTICATION_SECRETS = (WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY)
    ENABLED = TRUE;
GRANT USAGE ON INTEGRATION GOOGLE_MAPS_EAI TO ROLE SYSADMIN;
GRANT READ ON SECRET WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY TO ROLE SYSADMIN;
USE ROLE SYSADMIN;

In [ ]:
%%sql -r udf_google
-- Python UDF = your Python script, executed by Snowflake on each row. No server, no cron, no file I/O.
-- It returns a VARIANT (JSON) so we keep the full Google answer and parse it in SQL afterwards.
-- status values: OK / NOT_FOUND (no result) / ERROR (bad key, quota, API disabled...: see the error field).
CREATE OR REPLACE FUNCTION FIND_GOOGLE_PLACE(AGENCY_NAME VARCHAR, ADDRESS VARCHAR, CITY VARCHAR, ZIP_CODE VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
HANDLER = 'find_place'
EXTERNAL_ACCESS_INTEGRATIONS = (GOOGLE_MAPS_EAI)
SECRETS = ('gmap_key' = WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY)
AS
$$
import _snowflake
import requests

def find_place(agency_name, address, city, zip_code):
    key = _snowflake.get_generic_secret_string('gmap_key')
    query = ', '.join(p for p in [agency_name, address, zip_code, city] if p)
    try:
        r = requests.get(
            "https://maps.googleapis.com/maps/api/place/findplacefromtext/json",
            params={"input": query, "inputtype": "textquery",
                    "fields": "place_id", "language": "fr", "key": key},
            timeout=15)
        r.raise_for_status()
        data = r.json()
        if data.get("status") == "ZERO_RESULTS":
            return {"status": "NOT_FOUND", "query": query}
        if data.get("status") != "OK":
            # REQUEST_DENIED (bad key / API not enabled), OVER_QUERY_LIMIT, INVALID_REQUEST...
            return {"status": "ERROR", "query": query,
                    "error": f"{data.get('status')}: {data.get('error_message', '')}"}
        cands = data["candidates"]
        place_id = cands[0]["place_id"]
        d = requests.get(
            "https://maps.googleapis.com/maps/api/place/details/json",
            params={"place_id": place_id, "language": "fr", "key": key,
                    "fields": "place_id,name,formatted_address,formatted_phone_number,website,geometry,business_status,types"},
            timeout=15)
        d.raise_for_status()
        res = d.json().get("result", {})
        loc = res.get("geometry", {}).get("location", {})
        return {"status": "OK", "query": query, "google_place_id": place_id,
                "name": res.get("name"), "formatted_address": res.get("formatted_address"),
                "phone": res.get("formatted_phone_number"), "website": res.get("website"),
                "latitude": loc.get("lat"), "longitude": loc.get("lng"),
                "business_status": res.get("business_status"), "types": res.get("types", [])}
    except Exception as e:
        # type only: requests errors embed the URL, which contains the API key
        return {"status": "ERROR", "query": query, "error": type(e).__name__}
$$;

In [ ]:
%%sql -r test_google
-- Always test on a few rows first: each call is billed by Google.
SELECT AGENCY_NAME, FIND_GOOGLE_PLACE(AGENCY_NAME, ADDRESS, CITY, ZIP_CODE) AS GMAP
FROM DEALERS LIMIT 3;

Run the UDF once per dealer (SAMPLE_SIZE rows), keep the raw results, then fan out into `DEALERS.GOOGLE_PLACE_ID` and the `DEALER_GOOGLE_MAP` cache.

In [ ]:
%%sql -r gmap_results
-- Call the API ONCE per dealer and keep the raw results in a table.
-- The next cells reuse this table instead of calling the API again.
-- {{SAMPLE_SIZE}} is the Python variable defined at the top of the notebook (Jinja templating).
CREATE OR REPLACE TABLE GMAP_RESULTS AS
SELECT AGENCY_ID, SITE, FIND_GOOGLE_PLACE(AGENCY_NAME, ADDRESS, CITY, ZIP_CODE) AS GMAP
FROM (SELECT * FROM DEALERS ORDER BY AGENCY_ID LIMIT {{SAMPLE_SIZE}});

In [ ]:
%%sql -r gmap_status
-- Check before going further: if everything is ERROR, read the error field (usually the key or a disabled API).
SELECT GMAP:status::VARCHAR AS STATUS, COUNT(*) AS N FROM GMAP_RESULTS GROUP BY 1;

In [ ]:
%%sql -r fill_place_id
-- Store the Place ID on the dealer row (DEALERS.GOOGLE_PLACE_ID, as in the target architecture).
UPDATE DEALERS d
SET GOOGLE_PLACE_ID = g.GMAP:google_place_id::VARCHAR
FROM GMAP_RESULTS g
WHERE d.AGENCY_ID = g.AGENCY_ID AND d.SITE = g.SITE
  AND g.GMAP:status::VARCHAR = 'OK';

In [ ]:
%%sql -r fill_gmap_cache
-- Fill the cache. MERGE ... WHEN NOT MATCHED = only insert places we do not know yet,
-- so re-running the pipeline next month does not duplicate rows.
-- QUALIFY keeps one row per Place ID when several dealers point to the same place (multi-site dealers).
MERGE INTO DEALER_GOOGLE_MAP t
USING (
    SELECT GMAP:google_place_id::VARCHAR AS GOOGLE_PLACE_ID,
           GMAP:name::VARCHAR            AS AI_AGENCY_NAME,
           GMAP                          AS GOOGLE_JSON
    FROM GMAP_RESULTS
    WHERE GMAP:status::VARCHAR = 'OK'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY GOOGLE_PLACE_ID ORDER BY AGENCY_ID) = 1
) s
ON t.GOOGLE_PLACE_ID = s.GOOGLE_PLACE_ID
WHEN NOT MATCHED THEN INSERT (GOOGLE_PLACE_ID, AI_AGENCY_NAME, GOOGLE_JSON)
VALUES (s.GOOGLE_PLACE_ID, s.AI_AGENCY_NAME, s.GOOGLE_JSON);

In [ ]:
%%sql -r gmap_preview
-- Content of the Google Maps cache.
SELECT GOOGLE_PLACE_ID, AI_AGENCY_NAME,
       GOOGLE_JSON:formatted_address::VARCHAR AS ADDRESS,
       GOOGLE_JSON:phone::VARCHAR AS PHONE
FROM DEALER_GOOGLE_MAP;

## Block 3 — Enrich: Gemini search via External Access Integration
Uses **Joreca's production prompt** (`prompts/gemini_search_agency.txt` on the stage) with Google Search grounding, and the same `Agency Name` / `Address` inputs as their batch. Paste the Gemini key below.

Each call does web research: expect ~10-30 s per dealer. Token usage is kept in `AI_JSON:_usage`.

In [ ]:
%%sql -r eai_gemini
-- Same pattern as Google Maps: network rule + secret + integration, for the Gemini API host.
-- Requires ACCOUNTADMIN. Run once.
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE WORKSHOP_DB.PUBLIC.GEMINI_RULE
    MODE = EGRESS TYPE = HOST_PORT
    VALUE_LIST = ('generativelanguage.googleapis.com');
CREATE OR REPLACE SECRET WORKSHOP_DB.PUBLIC.GEMINI_API_KEY
    TYPE = GENERIC_STRING
    SECRET_STRING = 'PASTE_GEMINI_KEY_HERE';
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION GEMINI_EAI
    ALLOWED_NETWORK_RULES = (WORKSHOP_DB.PUBLIC.GEMINI_RULE)
    ALLOWED_AUTHENTICATION_SECRETS = (WORKSHOP_DB.PUBLIC.GEMINI_API_KEY)
    ENABLED = TRUE;
GRANT USAGE ON INTEGRATION GEMINI_EAI TO ROLE SYSADMIN;
GRANT READ ON SECRET WORKSHOP_DB.PUBLIC.GEMINI_API_KEY TO ROLE SYSADMIN;
USE ROLE SYSADMIN;

In [ ]:
%%sql -r udf_gemini
-- Replaces the Gemini batch flow (export JSON file -> upload -> wait -> download -> re-import).
-- Here each row calls Gemini directly and the answer lands in a table.
-- The prompt is Joreca's production prompt, read from a file on the stage (IMPORTS):
-- to change the prompt, re-upload the file and re-run this CREATE FUNCTION; no code change needed.
-- tools = google_search enables web research (grounding), as required by the prompt.
-- usageMetadata (input/output tokens) is kept in the result to measure the real cost per dealer.
CREATE OR REPLACE FUNCTION SEARCH_AGENCY(AGENCY_NAME VARCHAR, ADDRESS VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
IMPORTS = ('@WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE/prompts/gemini_search_agency.txt')
HANDLER = 'search'
EXTERNAL_ACCESS_INTEGRATIONS = (GEMINI_EAI)
SECRETS = ('gemini_key' = WORKSHOP_DB.PUBLIC.GEMINI_API_KEY)
AS
$$
import _snowflake
import json
import os
import re
import sys
import requests

MODEL = "gemini-2.5-flash"
# Joreca's production prompt, versioned as a file on the stage
with open(os.path.join(sys._xoptions["snowflake_import_directory"], "gemini_search_agency.txt"), encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

def _parse(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

def search(agency_name, address):
    key = _snowflake.get_generic_secret_string('gemini_key')
    user_input = f"Agency Name: {agency_name}\nAddress: {address}"
    body = {
        "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [{"role": "user", "parts": [{"text": user_input}]}],
        "tools": [{"google_search": {}}],          # online research, as in their batch setup
        "generationConfig": {"temperature": 0.1},
    }
    try:
        r = requests.post(
            f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
            headers={"x-goog-api-key": key}, json=body, timeout=120)
        if r.status_code != 200:
            return {"error": f"HTTP {r.status_code}: {r.json().get('error', {}).get('message', r.text[:300])}"}
        resp = r.json()
        text = "".join(p.get("text", "") for p in resp["candidates"][0]["content"]["parts"])
        out = _parse(text)
        out["_usage"] = resp.get("usageMetadata")   # input/output tokens, for cost tracking
        return out
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}
$$;

In [ ]:
%%sql -r test_gemini
-- Same inputs as the batch: 'Agency Name' and one 'Address' string.
-- Each call does web research: expect several seconds per dealer.
SELECT AGENCY_NAME,
       SEARCH_AGENCY(AGENCY_NAME, CONCAT_WS(', ', ADDRESS, ZIP_CODE, CITY, 'France')) AS AI
FROM DEALERS
LIMIT 3;

In [ ]:
%%sql -r fill_das
-- Run the search for SAMPLE_SIZE dealers and store the full JSON in AI_JSON.
-- AI_STANDARD_ADDRESS is extracted from potential_matches[0].standard_address (format defined in the prompt).
-- IS_REUSED = FALSE: in production, skip the call when last month already has a result for (AGENCY_ID, SITE).
INSERT INTO DEALER_ADDRESS_STANDARD (AGENCY_ID, SITE, AI_JSON, AI_STANDARD_ADDRESS, IS_REUSED)
SELECT AGENCY_ID, SITE, AI, AI:potential_matches[0]:standard_address::VARCHAR, FALSE
FROM (
    SELECT AGENCY_ID, SITE,
           SEARCH_AGENCY(AGENCY_NAME, CONCAT_WS(', ', ADDRESS, ZIP_CODE, CITY, 'France')) AS AI
    FROM (SELECT * FROM DEALERS ORDER BY AGENCY_ID LIMIT {{SAMPLE_SIZE}})
);

In [ ]:
%%sql -r das_preview
-- Parse the JSON with the : and [] notation. The ERROR column must be empty.
SELECT AGENCY_ID, SITE,
       AI_JSON:overall_search_status::VARCHAR                          AS SEARCH_STATUS,
       AI_JSON:potential_matches[0]:found_agency_name::VARCHAR         AS FOUND_NAME,
       AI_STANDARD_ADDRESS,
       AI_JSON:potential_matches[0]:establishment_level_number::VARCHAR AS AI_SIRET,
       AI_JSON:potential_matches[0]:match_confidence_score::FLOAT      AS CONFIDENCE,
       AI_JSON:_usage:promptTokenCount::NUMBER                         AS INPUT_TOKENS,
       AI_JSON:_usage:candidatesTokenCount::NUMBER                     AS OUTPUT_TOKENS,
       AI_JSON:error::VARCHAR                                          AS ERROR
FROM DEALER_ADDRESS_STANDARD;

In [ ]:
%%sql -r token_usage
-- Real token consumption per call: input to the Gemini vs Cortex AI cost comparison.
SELECT COUNT(*) AS CALLS,
       ROUND(AVG(AI_JSON:_usage:promptTokenCount::NUMBER))     AS AVG_INPUT_TOKENS,
       ROUND(AVG(AI_JSON:_usage:candidatesTokenCount::NUMBER)) AS AVG_OUTPUT_TOKENS,
       ROUND(AVG(AI_JSON:_usage:totalTokenCount::NUMBER))      AS AVG_TOTAL_TOKENS
FROM DEALER_ADDRESS_STANDARD
WHERE AI_JSON:error IS NULL;

## Block 4 — Resolve
Keep the Google Place ID from step 2 only when Gemini found a confident match (score >= 0.7) with the same postal code as the Google Maps address.

In [ ]:
%%sql -r resolve
-- Step 4 (Resolve): decide which Google Place ID is the right one for each dealer.
-- Simplified rule for the workshop: keep it only if Gemini is confident (>= 0.7)
-- AND the postal code found by Gemini appears in the Google Maps address.
-- Production adds a second Gemini pass for ambiguous cases.
-- Keep the Google Place ID found in step 2 only if Gemini found a confident match
-- whose postal code is the same as the Google Maps address.
UPDATE DEALER_ADDRESS_STANDARD das
SET GOOGLE_PLACE_ID = d.GOOGLE_PLACE_ID
FROM DEALERS d
JOIN DEALER_GOOGLE_MAP g ON d.GOOGLE_PLACE_ID = g.GOOGLE_PLACE_ID
WHERE das.AGENCY_ID = d.AGENCY_ID AND das.SITE = d.SITE
  AND das.AI_JSON:potential_matches[0]:match_confidence_score::FLOAT >= 0.7
  AND CONTAINS(g.GOOGLE_JSON:formatted_address::VARCHAR,
               das.AI_JSON:potential_matches[0]:cp::VARCHAR);

In [ ]:
%%sql -r enriched_view
-- One view that joins the 3 tables: raw crawl vs Gemini vs Google Maps, side by side.
CREATE OR REPLACE VIEW DEALERS_ENRICHED AS
SELECT d.AGENCY_ID, d.SITE,
       d.AGENCY_NAME                                                   AS RAW_NAME,
       das.AI_JSON:potential_matches[0]:found_agency_name::VARCHAR     AS AI_AGENCY_NAME,
       CONCAT_WS(', ', d.ADDRESS, d.ZIP_CODE, d.CITY)                  AS RAW_ADDRESS,
       das.AI_STANDARD_ADDRESS,
       d.SIRET                                                         AS RAW_SIRET,
       das.AI_JSON:potential_matches[0]:establishment_level_number::VARCHAR AS AI_SIRET,
       das.GOOGLE_PLACE_ID                                             AS RESOLVED_PLACE_ID,
       g.GOOGLE_JSON:formatted_address::VARCHAR                        AS GOOGLE_ADDRESS,
       g.GOOGLE_JSON:latitude::FLOAT                                   AS LATITUDE,
       g.GOOGLE_JSON:longitude::FLOAT                                  AS LONGITUDE,
       das.AI_JSON:potential_matches[0]:match_confidence_score::FLOAT  AS CONFIDENCE
FROM DEALERS d
JOIN DEALER_ADDRESS_STANDARD das ON d.AGENCY_ID = das.AGENCY_ID AND d.SITE = das.SITE
LEFT JOIN DEALER_GOOGLE_MAP g ON das.GOOGLE_PLACE_ID = g.GOOGLE_PLACE_ID;

In [ ]:
%%sql -r enriched_preview
-- Compare RAW_* columns with the enriched ones.
SELECT * FROM DEALERS_ENRICHED ORDER BY CONFIDENCE DESC;

In [ ]:
%%sql -r summary
-- How many dealers were standardized and resolved.
SELECT COUNT(*) AS PROCESSED,
       COUNT(AI_STANDARD_ADDRESS) AS STANDARDIZED,
       COUNT(RESOLVED_PLACE_ID) AS PLACE_ID_RESOLVED
FROM DEALERS_ENRICHED;

## Cleanup (after the test)
Uncomment to drop everything.

In [ ]:
%%sql -r cleanup
-- USE ROLE ACCOUNTADMIN;
-- DROP INTEGRATION IF EXISTS GOOGLE_MAPS_EAI;
-- DROP INTEGRATION IF EXISTS GEMINI_EAI;
-- DROP DATABASE IF EXISTS WORKSHOP_DB;
SELECT 'cleanup cell (commented out)' AS INFO;